# for example of rat_intravenous

In [ ]:
import pandas as pd
from typing import Dict

spectra_csv = r"with_canonical_inchikey.csv"       
ld50_csv = r"rat_intravenous.csv"                     
output_csv = r"rat_intravenous_merged.csv"                
chunksize = 200000 

def normalize_colname(name: str) -> str:
    return name.strip().lower().replace('_', ' ').replace('-', ' ')

def find_col(cols, target_names):
    norm_map = {normalize_colname(c): c for c in cols}
    for t in target_names:
        tnorm = normalize_colname(t)
        if tnorm in norm_map:
            return norm_map[tnorm]
    return None

def build_ld50_index(ld50_csv: str, chunksize=200000):
    ld_by_fullik: Dict[str, dict] = {}
    ld_by_key14: Dict[str, dict] = {}
    ld_by_can: Dict[str, dict] = {}

    df_head = pd.read_csv(ld50_csv, nrows=0)
    cols = list(df_head.columns)

    ik_col = find_col(cols, ["InChIKey", "inchikey"])
    can_col = find_col(cols, ["Canonical SMILES", "canonical_smiles", "canonical smiles"])

    required_cols = ["TAID","Pubchem CID","IUPAC Name","SMILES","Canonical SMILES","InChIKey","Toxicity Value"]
    col_map = {}
    for rc in required_cols:
        found = find_col(cols, [rc, rc.replace(" ", "_")])
        col_map[rc] = found

    for chunk in pd.read_csv(ld50_csv, chunksize=chunksize, dtype=str):
        for _, row in chunk.iterrows():
            ik = str(row.get(ik_col, "")).strip() if ik_col else None
            can = str(row.get(can_col, "")).strip() if can_col else None
            if ik in ["", "nan", "None"]: ik = None
            if can in ["", "nan", "None"]: can = None

            out_row = {}
            for rc, mapped in col_map.items():
                out_row[rc] = row.get(mapped, "") if (mapped in row and pd.notna(row.get(mapped))) else ""

            if ik:
                ld_by_fullik[ik] = out_row
                ld_by_key14[ik[:14]] = out_row
            if can:
                ld_by_can[can] = out_row

    return ld_by_fullik, ld_by_key14, ld_by_can


def merge_with_key14(spectra_csv: str, ld50_csv: str, output_csv: str, chunksize: int = 200000):
    ld_by_fullik, ld_by_key14, ld_by_can = build_ld50_index(ld50_csv, chunksize)

    out_cols = ["SPECTRUMID","TAID","Pubchem CID","IUPAC Name","SMILES",
                "Canonical SMILES","InChIKey","Toxicity Value","Match_Type"]
    all_rows = []

    for chunk in pd.read_csv(spectra_csv, chunksize=chunksize, dtype=str):
        cols = list(chunk.columns)
        ik_col = find_col(cols, ["InChIKey", "inchikey"])
        can_col = find_col(cols, ["Canonical SMILES", "canonical_smiles", "canonical smiles"])
        sid_col = find_col(cols, ["SPECTRUMID", "spectrum id", "spectrumid"])

        for _, row in chunk.iterrows():
            matched = None
            match_type = None

            # 1️⃣ full InChIKey 
            if ik_col and pd.notna(row.get(ik_col, None)):
                ik = str(row[ik_col]).strip()
                if ik in ld_by_fullik:
                    matched = ld_by_fullik[ik]
                    match_type = "Full_InChIKey"

            # 2️⃣ key14 
            if matched is None and ik_col and pd.notna(row.get(ik_col, None)):
                ik14 = str(row[ik_col]).strip()[:14]
                if ik14 in ld_by_key14:
                    matched = ld_by_key14[ik14]
                    match_type = "Key14_InChIKey"

            # 3️⃣ Canonical SMILES 
            if matched is None and can_col and pd.notna(row.get(can_col, None)):
                can = str(row[can_col]).strip()
                if can in ld_by_can:
                    matched = ld_by_can[can]
                    match_type = "Canonical_SMILES"

            if matched:
                out_row = {k: matched.get(k, "") for k in out_cols if k in matched}
                out_row["SPECTRUMID"] = row.get(sid_col, "") if sid_col else ""
                out_row["Match_Type"] = match_type
                all_rows.append(out_row)

    df = pd.DataFrame(all_rows)

    before = len(df)
    df['dedup_key'] = df['InChIKey'].fillna('')
    mask_empty = df['dedup_key'] == ''
    df.loc[mask_empty, 'dedup_key'] = df.loc[mask_empty, 'Canonical SMILES'].fillna('')
    df = df.drop_duplicates(subset='dedup_key', keep='first').drop(columns='dedup_key')
    after = len(df)
    print(f"[INFO] deduplicated {before - after} duplicate records, keeping {after} unique compounds")

    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"[DONE] Output file created: {output_csv}")

merge_with_key14(spectra_csv, ld50_csv, output_csv, chunksize)


[INFO] 已建立 LD50 索引：2321 full InChIKey，2321 key14，2323 Canonical SMILES
[INFO] 匹配完成，共 49622 条记录
[INFO] 去重完成：删除 48621 条重复记录，保留 1001 条唯一化合物
[DONE] 已输出文件：E:\LD\csv\rat\1rat_intravenous_merged.csv
